In [1]:
import pandas as pd
from sqlalchemy import create_engine

In [2]:
engine = create_engine('sqlite:///market_data.db')
df_stocks = pd.read_sql('stock_prices', engine)
df_news = pd.read_sql('news_sentiment', engine)

In [3]:
df_stocks['Date'] = pd.to_datetime(df_stocks['Date'], utc=True).dt.normalize()
df_news['Date'] = pd.to_datetime(df_news['Date'], utc=True).dt.normalize()

In [4]:
df_news_daily = df_news.sort_values('Confidence', ascending=False).drop_duplicates(subset=['Date', 'Ticker'])

In [5]:
df_news['Date'] = df_news['Date'].apply(
    lambda x: x + pd.Timedelta(days=7-x.weekday()) if x.weekday() >= 5 else x
)

In [6]:
sentiment_map = {'positive': 1, 'neutral': 0, 'negative': -1}
df_news['Sentiment_Score'] = df_news['Sentiment'].map(sentiment_map) * df_news['Confidence']

In [7]:
df_news_daily = df_news.groupby(['Date', 'Ticker']).agg(
    Article_Count=('Headline', 'count'),
    Avg_Sentiment_Score=('Sentiment_Score', 'mean'),
    Dominant_Sentiment=('Sentiment', lambda x: x.mode()[0] if not x.empty else 'neutral')
).reset_index()

In [8]:
df_merged = pd.merge(df_stocks, df_news_daily, on=['Date', 'Ticker'], how='left')

In [9]:
df_merged['Article_Count'] = df_merged['Article_Count'].fillna(0)
df_merged['Avg_Sentiment_Score'] = df_merged['Avg_Sentiment_Score'].fillna(0)
df_merged['Dominant_Sentiment'] = df_merged['Dominant_Sentiment'].fillna('No News')

In [10]:
df_merged = df_merged.sort_values(by=['Ticker', 'Date'])

In [11]:
df_merged['Daily_Return'] = df_merged.groupby('Ticker')['Close'].pct_change()

df_merged['Daily_Return'] = df_merged['Daily_Return'].fillna(0)

In [12]:
df_merged.to_sql('analytical_view', con=engine, if_exists='replace', index=False)

5285

In [ ]:
print(df_merged[df_merged['Article_Count'] > 0][['Date', 'Ticker', 'Close', 'Article_Count', 'Avg_Sentiment_Score', 'Dominant_Sentiment', 'Daily_Return']].head(10))